---
## Step 1 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error

---
## Step 2 — Load Dataset

We load the **California Housing Dataset** directly from scikit-learn.  
It contains **20,640 records** from the 1990 California census with **8 features** and 1 target variable (`MedHouseVal`).

In [ ]:
# Load dataset as a DataFrame
housing = fetch_california_housing(as_frame=True)
mydata = housing.data
mytarget = housing.target

# Merge features and target into one DataFrame
dataset = mydata.join(mytarget)
dataset.head()

---
## Step 3 — Exploratory Data Analysis (EDA)

### 3.1 Dataset Shape & Info

In [ ]:
print(f"Shape: {dataset.shape}")
print("\n--- Dataset Info ---")
dataset.info()

### 3.2 Basic Statistics

In [ ]:
dataset.describe()

### 3.3 Missing Values Check

In [ ]:
print("Missing values per column:")
print(dataset.isnull().sum())
print("\nNo missing values found!" if dataset.isnull().sum().sum() == 0 else "\nMissing values found!")

### 3.4 Target Variable Distribution

The target `MedHouseVal` is **right-skewed** — most houses are in the \$100K–\$300K range, with a cap at \$500K.

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(dataset['MedHouseVal'], bins=50, color='steelblue', edgecolor='white')
plt.xlabel("Median House Value ($100,000s)")
plt.ylabel("Frequency")
plt.title("Distribution of Median House Values")
plt.tight_layout()
plt.show()

### 3.5 Correlation Heatmap

Key finding: **MedInc (Median Income)** has the strongest positive correlation (+0.69) with `MedHouseVal`.

In [ ]:
plt.figure(figsize=(10, 8))
corr = dataset.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

---
## Step 4 — Preprocessing & Train/Test Split

- **X** = All 8 features (drop target column)
- **y** = Target column (`MedHouseVal`)
- **Split:** 80% Train / 20% Test with `random_state=42` for reproducibility

In [ ]:
# Separate features and target
X = dataset.drop('MedHouseVal', axis=1)
y = dataset['MedHouseVal']

# Train/Test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

---
##  Step 5 — Model Training

We use **Linear Regression** from scikit-learn. The model learns a weight for each feature to minimize the Sum of Squared Errors between predictions and actual values.

> `Price = w1×MedInc + w2×HouseAge + w3×AveRooms + ... + bias`

In [ ]:
# Initialize and train the model
lr = LinearRegression()
lr.fit(X_train, y_train)

# Generate predictions on test set
y_pred = lr.predict(X_test)

print("Model trained successfully!")

---
## Step 6 — Model Evaluation

| Metric | Meaning |
|--------|----------|
| **MAE** | Average prediction error (in $100,000s) |
| **RMSE** | Penalizes large errors more than MAE |
| **R²** | % of price variation explained by the model |

In [ ]:
mae  = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.4f}  → Predictions off by ~${mae*100000:,.0f} on average")
print(f"RMSE : {rmse:.4f}  → Larger errors present")
print(f"R²   : {r2:.4f}  → Model explains {r2*100:.1f}% of price variation")

---
## Step 7 — Visualizations

### 7.1 Actual vs Predicted Values

Points close to the **red diagonal line** = good predictions.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.4, color='steelblue', s=10)
plt.plot([0, 6], [0, 6], 'r-', linewidth=2, label='Perfect Prediction')
plt.xlabel("Actual Values ($100,000s)")
plt.ylabel("Predicted Values ($100,000s)")
plt.title("Actual vs Predicted House Values")
plt.legend()
plt.tight_layout()
plt.show()

### 7.2 Residuals Plot

Residuals should be **randomly scattered around 0** for a good model.  
A downward trend here indicates the model **underestimates high-priced homes** — a known limitation of Linear Regression.

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.4, color='steelblue', s=10)
plt.axhline(y=0, color='red', linestyle='--', linewidth=2, label='Zero Error Line')
plt.xlabel("Predicted Values ($100,000s)")
plt.ylabel("Residuals")
plt.title("Residuals Plot")
plt.legend()
plt.tight_layout()
plt.show()

---
## Step 8 — Save Model

We save the trained model using **joblib** for future use or deployment.

In [ ]:
model_filename = 'california_housing_lr.joblib'
joblib.dump(lr, model_filename)
print(f" Model saved as '{model_filename}'")

---
## Summary

| Item | Value |
|------|-------|
| Dataset | California Housing (20,640 rows, 8 features) |
| Algorithm | Linear Regression |
| Train/Test Split | 80% / 20% |
| MAE | ~0.5332 (~$53,320 avg error) |
| RMSE | ~0.7456 |
| R² Score | ~0.5758 (57.58% variance explained) |

### Improvement Ideas
- **Feature Engineering:** Create `rooms_per_person = AveRooms / AveOccup`
- **Regularization:** Try `Ridge` or `Lasso` Regression
- **Better Models:** `RandomForestRegressor` can push R² above 0.80
- **Feature Scaling:** Apply `StandardScaler` before training
- **Cross Validation:** Use k-fold CV for more reliable evaluation